In [12]:
import numpy as np
import pandas as pd
from collections import defaultdict

# ============ CONFIGURATION ============
# Set your file paths here
NPY_FILE_PATH = r'C:/Users/Zaria/OneDrive - University of Illinois Chicago/Desktop/CD3/CD3_merged.npy'  # Your nupy file with ephys data
EXCEL_FILE_PATH = 'B:/TonesPlaylist_Individual.csv'  # Your Excel file with playlist info
OUTPUT_FILE_PATH = r'B:/CD3.Processed/CD3.redo.All.Units.BF.NEWWW.npy'  # Output file name

# Set your playlist start time (in seconds)
PLAYLIST_START_TIME = 2221.658   # CHANGE THIS to your actual playlist start time
#norm is 2221.658
# Stimulus duration in seconds
STIMULUS_DURATION = 0.1  # 100 ms

# ============ LOAD DATA ============
print("Loading data...")
loaded_data = np.load(NPY_FILE_PATH, allow_pickle=True).item()

time_intervals = loaded_data['time_intervals']
spike_times_dict = loaded_data['spike_times_dict']
waveforms = loaded_data.get('waveforms_to_plot', None)  # Use .get() in case it doesn't exist
sampling_rate = loaded_data.get('sampling_rate', None)

print("Loading playlist data...")
playlist_df = pd.read_csv(EXCEL_FILE_PATH)
print(f"Loaded {len(playlist_df)} stimuli from playlist")

# ============ USE FILENAME AS IDENTIFIER ============
print("Using filenames as identifiers...")

# Since filenames are already the identifiers, just use them directly
# Remove any NaN values just in case
playlist_df = playlist_df.dropna(subset=['Filename'])
playlist_df['identifier'] = playlist_df['Filename']

print(f"Found {playlist_df['identifier'].nunique()} unique stimulus types")
print(f"Unique identifiers: {sorted(playlist_df['identifier'].unique())}")

# ============ CALCULATE FIRING RATES FOR EACH UNIT ============
print("\nCalculating firing rates for each unit...")

# Dictionary to store results for each unit
unit_firing_rates = {}  # {unit_id: {identifier: [rates]}}
unit_best_frequency = {}

for i, unit_id in enumerate(spike_times_dict.keys()):
    print(f"Processing unit {i+1}/{len(spike_times_dict)}: {unit_id}")
    
    unit_spikes = np.array(spike_times_dict[unit_id])
    unit_firing_rates[unit_id] = defaultdict(list)
    
    # For each stimulus presentation in the playlist
    for idx, row in playlist_df.iterrows():
        identifier = row['identifier']
        stim_offset = row['Time']  # Time from playlist start
        
        # Calculate absolute time window for this stimulus
        stim_start = PLAYLIST_START_TIME + stim_offset
        stim_end = stim_start + STIMULUS_DURATION
        
        # Count spikes in this window
        spikes_in_window = np.sum((unit_spikes >= stim_start) & (unit_spikes < stim_end))
        
        # Calculate firing rate (spikes per second)
        firing_rate = spikes_in_window / STIMULUS_DURATION
        
        unit_firing_rates[unit_id][identifier].append(firing_rate)
    
    # Calculate average firing rate for each identifier
    avg_firing_rates = {}
    for identifier, rates in unit_firing_rates[unit_id].items():
        avg_firing_rates[identifier] = np.mean(rates)
    
    # Find best frequency (identifier with highest average firing rate)
    if avg_firing_rates:
        best_freq = max(avg_firing_rates, key=avg_firing_rates.get)
        unit_best_frequency[unit_id] = best_freq
        print(f"  Best frequency: {best_freq} (avg rate: {avg_firing_rates[best_freq]:.2f} Hz)")
    else:
        unit_best_frequency[unit_id] = None
        print(f"  No activity detected")

# ============ PREPARE OUTPUT DATA ============
print("\nPreparing output data...")

# Convert firing rates to a more compact format
# Create arrays for avg firing rates per identifier for each unit
unique_identifiers = sorted(playlist_df['identifier'].unique())

unit_avg_firing_rates = {}
for unit_id in spike_times_dict.keys():
    rates = []
    for identifier in unique_identifiers:
        if identifier in unit_firing_rates[unit_id]:
            avg_rate = np.mean(unit_firing_rates[unit_id][identifier])
        else:
            avg_rate = 0.0
        rates.append(avg_rate)
    unit_avg_firing_rates[unit_id] = np.array(rates)

# ============ SAVE ENHANCED DATA ============
print("Saving enhanced data...")

output_data = {
    # Original data
    'time_intervals': time_intervals,
    'spike_times_dict': spike_times_dict,
    'waveforms': waveforms,
    'sampling_rate': sampling_rate,
    
    # New analysis data
    'playlist_start_time': PLAYLIST_START_TIME,
    'stimulus_identifiers': unique_identifiers,  # List of all unique identifiers
    'unit_best_frequency': unit_best_frequency,  # {unit_id: best_identifier}
    'unit_avg_firing_rates': unit_avg_firing_rates,  # {unit_id: array of avg rates per identifier}
    'unit_all_firing_rates': dict(unit_firing_rates),  # Full data: {unit_id: {identifier: [rates]}}
    'playlist_info': playlist_df.to_dict('records')  # Complete playlist information
}

np.save(OUTPUT_FILE_PATH, output_data)
print(f"\nData saved to: {OUTPUT_FILE_PATH}")

# ============ SUMMARY ============
print("\n" + "="*50)
print("ANALYSIS SUMMARY")
print("="*50)
print(f"Total units analyzed: {len(spike_times_dict)}")
print(f"Unique stimulus types: {len(unique_identifiers)}")
print(f"Total stimulus presentations: {len(playlist_df)}")
print("\nBest frequencies by unit:")
for unit_id, best_freq in unit_best_frequency.items():
    print(f"  {unit_id}: {best_freq}")

print("\n" + "="*50)
print("OUTPUT DATA STRUCTURE")
print("="*50)
print("The saved .npy file contains:")
print("  - All original data (time_intervals, spike_times_dict, waveforms, sampling_rate)")
print("  - playlist_start_time: The start time you specified")
print("  - stimulus_identifiers: List of unique stimulus identifiers")
print("  - unit_best_frequency: Dictionary mapping each unit to its best frequency")
print("  - unit_avg_firing_rates: Dictionary with average firing rates per stimulus for each unit")
print("  - unit_all_firing_rates: Complete firing rate data for all presentations")
print("  - playlist_info: Complete playlist information from Excel")

Loading data...
Loading playlist data...
Loaded 1201 stimuli from playlist
Using filenames as identifiers...
Found 120 unique stimulus types
Unique identifiers: ['100_kHz_100_dB', '100_kHz_10_dB', '100_kHz_20_dB', '100_kHz_30_dB', '100_kHz_40_dB', '100_kHz_50_dB', '100_kHz_60_dB', '100_kHz_70_dB', '100_kHz_80_dB', '100_kHz_90_dB', '10_kHz_100_dB', '10_kHz_10_dB', '10_kHz_20_dB', '10_kHz_30_dB', '10_kHz_40_dB', '10_kHz_50_dB', '10_kHz_60_dB', '10_kHz_70_dB', '10_kHz_80_dB', '10_kHz_90_dB', '110_kHz_100_dB', '110_kHz_10_dB', '110_kHz_20_dB', '110_kHz_30_dB', '110_kHz_40_dB', '110_kHz_50_dB', '110_kHz_60_dB', '110_kHz_70_dB', '110_kHz_80_dB', '110_kHz_90_dB', '120_kHz_100_dB', '120_kHz_10_dB', '120_kHz_20_dB', '120_kHz_30_dB', '120_kHz_40_dB', '120_kHz_50_dB', '120_kHz_60_dB', '120_kHz_70_dB', '120_kHz_80_dB', '120_kHz_90_dB', '20_kHz_100_dB', '20_kHz_10_dB', '20_kHz_20_dB', '20_kHz_30_dB', '20_kHz_40_dB', '20_kHz_50_dB', '20_kHz_60_dB', '20_kHz_70_dB', '20_kHz_80_dB', '20_kHz_90_dB', '30

In [ ]:
#Cleaning units with 0Hz
# 
# import numpy as np
import numpy as np
import pandas as pd
from collections import defaultdict

# ============ CONFIGURATION ============
INPUT_FILE_PATH = r'B:/CD3.Processed/CD3.redo.All.Units.BF.NEWWW.npy'  # Input file with BF analysis
OUTPUT_FILE_PATH = r'B:/CD3.Processed/CD3.redo.All.Units.BF.Filtered.npy'  # Output file (non-zero units only)

# ============ LOAD DATA ============
print("Loading data...")
loaded_data = np.load(INPUT_FILE_PATH, allow_pickle=True).item()

# Extract all the data
time_intervals = loaded_data['time_intervals']
spike_times_dict = loaded_data['spike_times_dict']
waveforms = loaded_data.get('waveforms', None)
sampling_rate = loaded_data.get('sampling_rate', None)
playlist_start_time = loaded_data['playlist_start_time']
stimulus_identifiers = loaded_data['stimulus_identifiers']
unit_best_frequency = loaded_data['unit_best_frequency']
unit_avg_firing_rates = loaded_data['unit_avg_firing_rates']
unit_all_firing_rates = loaded_data['unit_all_firing_rates']
playlist_info = loaded_data['playlist_info']

print(f"Loaded data with {len(spike_times_dict)} total units")

# ============ FILTER UNITS ============
print("\nFiltering units with non-zero best frequencies...")

# Find units where the best frequency has an average firing rate > 0
units_to_keep = []
units_to_remove = []

for unit_id in spike_times_dict.keys():
    best_freq = unit_best_frequency[unit_id]
    
    if best_freq is None:
        # No best frequency (no activity at all)
        units_to_remove.append(unit_id)
        print(f"  Removing {unit_id}: No activity detected")
    else:
        # Check if the best frequency has a firing rate > 0
        best_freq_index = stimulus_identifiers.index(best_freq)
        best_freq_rate = unit_avg_firing_rates[unit_id][best_freq_index]
        
        if best_freq_rate > 0:
            units_to_keep.append(unit_id)
        else:
            units_to_remove.append(unit_id)
            print(f"  Removing {unit_id}: Best frequency {best_freq} has 0 Hz firing rate")

print(f"\nUnits to keep: {len(units_to_keep)}")
print(f"Units to remove: {len(units_to_remove)}")

# ============ CREATE FILTERED DICTIONARIES ============
print("\nCreating filtered data structures...")

# Filter spike_times_dict
filtered_spike_times_dict = {uid: spike_times_dict[uid] for uid in units_to_keep}

# Filter waveforms if it exists
filtered_waveforms = None
if waveforms is not None:
    if isinstance(waveforms, dict):
        filtered_waveforms = {uid: waveforms[uid] for uid in units_to_keep if uid in waveforms}
    else:
        filtered_waveforms = waveforms  # Keep as is if not a dict

# Filter unit_best_frequency
filtered_unit_best_frequency = {uid: unit_best_frequency[uid] for uid in units_to_keep}

# Filter unit_avg_firing_rates
filtered_unit_avg_firing_rates = {uid: unit_avg_firing_rates[uid] for uid in units_to_keep}

# Filter unit_all_firing_rates
filtered_unit_all_firing_rates = {uid: unit_all_firing_rates[uid] for uid in units_to_keep}

# ============ SAVE FILTERED DATA ============
print("Saving filtered data...")

output_data = {
    # Original data (filtered)
    'time_intervals': time_intervals,
    'spike_times_dict': filtered_spike_times_dict,
    'waveforms': filtered_waveforms,
    'sampling_rate': sampling_rate,
    
    # Analysis data (filtered)
    'playlist_start_time': playlist_start_time,
    'stimulus_identifiers': stimulus_identifiers,  # Same stimulus list
    'unit_best_frequency': filtered_unit_best_frequency,
    'unit_avg_firing_rates': filtered_unit_avg_firing_rates,
    'unit_all_firing_rates': filtered_unit_all_firing_rates,
    'playlist_info': playlist_info,  # Same playlist info
    
    # Metadata about filtering
    'filtering_info': {
        'original_unit_count': len(spike_times_dict),
        'filtered_unit_count': len(filtered_spike_times_dict),
        'removed_unit_count': len(units_to_remove),
        'removed_units': units_to_remove,
        'filter_criterion': 'Removed units with best frequency firing rate = 0 Hz'
    }
}

np.save(OUTPUT_FILE_PATH, output_data)
print(f"\nFiltered data saved to: {OUTPUT_FILE_PATH}")

# ============ SUMMARY ============
print("\n" + "="*50)
print("FILTERING SUMMARY")
print("="*50)
print(f"Original units: {len(spike_times_dict)}")
print(f"Filtered units (kept): {len(filtered_spike_times_dict)}")
print(f"Removed units: {len(units_to_remove)}")
print(f"Retention rate: {100 * len(units_to_keep) / len(spike_times_dict):.1f}%")

print("\nRemoved units:")
for unit_id in units_to_remove:
    best_freq = unit_best_frequency[unit_id]
    if best_freq:
        print(f"  {unit_id}: Best frequency was {best_freq}")
    else:
        print(f"  {unit_id}: No activity detected")

print("\nKept units and their best frequencies:")
for unit_id in units_to_keep:
    best_freq = filtered_unit_best_frequency[unit_id]
    best_freq_index = stimulus_identifiers.index(best_freq)
    best_freq_rate = filtered_unit_avg_firing_rates[unit_id][best_freq_index]
    print(f"  {unit_id}: {best_freq} ({best_freq_rate:.2f} Hz)")

print("\n" + "="*50)

Loading data...
Loaded data with 370 total units

Filtering units with non-zero best frequencies...
  Removing 27: Best frequency 60_kHz_20_dB has 0 Hz firing rate
  Removing 29: Best frequency 60_kHz_20_dB has 0 Hz firing rate
  Removing 36: Best frequency 60_kHz_20_dB has 0 Hz firing rate
  Removing 41: Best frequency 60_kHz_20_dB has 0 Hz firing rate
  Removing 52: Best frequency 60_kHz_20_dB has 0 Hz firing rate
  Removing 64: Best frequency 60_kHz_20_dB has 0 Hz firing rate
  Removing 69: Best frequency 60_kHz_20_dB has 0 Hz firing rate
  Removing 70: Best frequency 60_kHz_20_dB has 0 Hz firing rate
  Removing 73: Best frequency 60_kHz_20_dB has 0 Hz firing rate
  Removing 74: Best frequency 60_kHz_20_dB has 0 Hz firing rate
  Removing 108: Best frequency 60_kHz_20_dB has 0 Hz firing rate
  Removing 112: Best frequency 60_kHz_20_dB has 0 Hz firing rate
  Removing 121: Best frequency 60_kHz_20_dB has 0 Hz firing rate
  Removing 122: Best frequency 60_kHz_20_dB has 0 Hz firing rate
